# Training LGBM+NN Hybrid Model

Changes made to improve efficiency:
1. Move lgb.Dataset construction to outside the quantile loop so it's done 28x per chunk instead of 252x
2. Change LGBM to use GPU
3. Move target creation and encoding to preprocess so it's done once instead of 55x
4. Pre-train the neural network on the full dataset first, freeze it, then train LGBM
- ML problem: NN was previously trained on only 1 gradient step per chunk of 1M rows. This means the embeddings are not meaningful (almost random) for the first few chunks
5. Use a fixed tree budget (500) split across chunks, so the model is not forced to build 200 trees every time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ADL Group Project

Mounted at /content/drive
/content/drive/MyDrive/ADL Group Project


In [ ]:
import os
os.chdir("/content/drive/MyDrive/ADL Group Project")

In [ ]:
import json
import os
import pickle
import random
import time

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

from base_model import BaseModel

In [ ]:
!nvidia-smi | head -20

Tue Apr  7 04:14:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   26C    P0             66W /  700W |       4MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
################ CREATING LIGHTGBM NN HYBRID MODEL #################

"""
The Neural Network is used to learn better feature representations (embeddings) of each time series, something a LightGBM cannot do.
These embeddings are used to create new feature matrices and become additional input features for the LightGBM to use

"""
class FeatureNN(nn.Module):
    def __init__(self, n_items, input_dim):
        super().__init__()
        self.emb = nn.Embedding(n_items, 64)

        self.backbone = nn.Sequential(
            nn.Linear(input_dim + 64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        # predicts a SINGLE value instead of predicting quantiles which is what LightGBM does
        # but even then we don't really use this output, only the representations it has used along the way
        # the NN does NOT forecast, just learns good representations which is why we are using MSE loss here instead of pinball loss
        self.head = nn.Linear(128, 1)

    def forward(self, x, item):
        x = torch.cat([x, self.emb(item)], dim=1)
        features = self.backbone(x)
        out = self.head(features)
        return out.squeeze(-1), features

class LightGBM_NN(BaseModel):

    @property
    def model_name(self):
        return "LightGBM_NN_Hybrid"

    def preprocess(self):
        from pandas.api.types import is_numeric_dtype
        print("starting preprocessing")
        # LightGBM cannot pick up time-series features easily so we must create these features
        # The NN will also use these features to help understand the time series patterns
        def add_features(df):
                df = df.sort_values(["id","d_num"])
                grp = df.groupby("id")["sales"]
                # creating previous sales at t-1, t-7, t-14, t-28 days
                # to learn past sale demand
                for lag in [1,7,14,28]:
                    df[f"lag_{lag}"] = grp.shift(lag)
                # average sales over last 7 and 28 days
                for w in [7,28]:
                    df[f"roll_mean_{w}"] = grp.shift(1).rolling(w).mean()
                # to model long term growth
                df["trend"] = df["d_num"].astype(np.int16)
                # to help with seasonality
                df["week_of_year"] = df["wm_yr_wk"].astype(np.int16)
                # transforms day of week into a circle to visualise where we are in the week better
                # this will help then to predict sales
                df["dow_sin"] = np.sin(2*np.pi*df["wday"]/7).astype(np.float32)
                df["dow_cos"] = np.cos(2*np.pi*df["wday"]/7).astype(np.float32)

                return df
        # add these features into the datasets
        train = add_features(self.train_raw)
        val = add_features(self.val_raw)
        test = add_features(self.test_raw)

        # fix: move create_targets, encode_categories and clean_numerics to preprocess
        cat_cols = ["item_id","dept_id","cat_id","store_id","state_id"]
        cat_maps = {c: {k:i for i,k in enumerate(train[c].unique())} for c in cat_cols}
        self.cat_maps = cat_maps

        def create_targets(df, horizon=28):
            df = df.copy()
            for h in range(1, horizon+1):
                df[f"target_{h}"] = df.groupby("id")["sales"].shift(-h)
            return df

        def encode_categories(df):
            for c in cat_cols:
                df[c] = df[c].map(cat_maps[c]).astype(np.int16)
            return df

        def clean_numerics(df):
            for col in df.select_dtypes(include=[np.number]).columns:
                df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
            return df

        train = create_targets(train)
        train = encode_categories(train)
        train = clean_numerics(train)
        val   = create_targets(val)
        val   = encode_categories(val)
        val   = clean_numerics(val)
        test  = encode_categories(test)
        test  = clean_numerics(test)

        # looping over the datasets to apply the same preprocessing
        for dataset in [train, val, test]:
            # get the IDs / categorical variables
            cat_cols = ["item_id","dept_id","cat_id","store_id","state_id"]
            # define the numerical features (everything except whats not in cat_cols)
            num_cols = [c for c in dataset.columns if c not in cat_cols]
            # loop through numerical columns
            for col in num_cols:
                col_data = dataset[col]
                # check they are actually numerical & converts NaNs to 0
                if is_numeric_dtype(col_data):
                    col_data = col_data.mask(np.isinf(col_data), np.nan)
                    col_data = col_data.fillna(0)
                    # note we are using float 32 to reduce memory usage
                    dataset[col] = col_data.astype(np.float32)
                else:
                    dataset[col] = col_data

        self.train_processed = train
        self.val_processed = val
        self.test_processed = test

    def _pretrain_nn(self, model_nn, scaler, features, target_cols, n_epochs=3):
        opt      = torch.optim.Adam(model_nn.parameters(), lr=0.0002)
        df_all   = self.train_processed
        chunk_size = 1_000_000
        model_nn.train()
        for epoch in range(n_epochs):
            total_loss, n_chunks = 0.0, 0
            for i in range(0, len(df_all), chunk_size):
                df = df_all.iloc[i:i+chunk_size].dropna(subset=target_cols)
                if len(df) == 0:
                    continue
                X      = scaler.transform(df[features].values.astype(np.float32))
                X_t    = torch.tensor(X,                    dtype=torch.float32).to(self.device)
                item_t = torch.tensor(df["item_id"].values, dtype=torch.long   ).to(self.device)
                y_nn   = torch.tensor(
                    df[target_cols].mean(axis=1).values.astype(np.float32)
                ).to(self.device)
                pred, _ = model_nn(X_t, item_t)
                loss    = nn.MSELoss()(pred, y_nn)
                opt.zero_grad(); loss.backward(); opt.step()
                total_loss += loss.item(); n_chunks += 1
            print(f"  NN pretrain epoch {epoch+1}/{n_epochs} | avg loss {total_loss/max(n_chunks,1):.4f}")
        for p in model_nn.parameters():
            p.requires_grad_(False)
        model_nn.eval()
        print("NN frozen.")
        return model_nn


    def train(self):
        import time
        from sklearn.preprocessing import StandardScaler
        import lightgbm as lgb

        print("\nTRAINING STARTING\n")

        quantiles = self.QUANTILES
        models = {q: None for q in quantiles}

        # NEW: define target_cols once here instead of redefining inside loops
        target_cols = [f"target_{h}" for h in range(1, 29)]

        checkpoint_path = f"{self.output_dir}/checkpoint_chunk.pkl"
        start_chunk = 0

        df_all = self.train_processed

        # REMOVED: cat_cols, cat_maps, n_items, encode_categories, create_targets
        # These are now computed once in preprocess() and stored in self.cat_maps

        chunk_size   = 1_000_000
        total_rows   = len(df_all)
        total_chunks = int(np.ceil(total_rows / chunk_size))

        # NEW: fixed tree budget split evenly across chunks (~9 rounds/chunk for 55 chunks)
        total_tree_budget = 500
        rounds_per_chunk  = max(1, total_tree_budget // total_chunks)

        print(f"Total chunks: {total_chunks} | Rounds per chunk: {rounds_per_chunk}")

        if os.path.exists(checkpoint_path):
            print("Found checkpoint, resuming training")
            start_chunk = joblib.load(checkpoint_path) + 1
            for q in quantiles:
                models[q] = lgb.Booster(model_file=f"{self.output_dir}/checkpoint_lgb_q_{q}.txt")
            scaler   = joblib.load(f"{self.output_dir}/checkpoint_scaler.pkl")
            features = joblib.load(f"{self.output_dir}/checkpoint_features.pkl")
            n_items  = joblib.load(f"{self.output_dir}/checkpoint_n_items.pkl")

        if not os.path.exists(checkpoint_path):
            print("Fitting scaler...")
            scaler   = StandardScaler()
            features = None

            for i in range(0, total_rows, chunk_size):
                # REMOVED: create_targets(df), encode_categories(df), manual inf/NaN cleaning
                # These are already done in preprocess() — df_all is already clean
                df = df_all.iloc[i:i+chunk_size].dropna(subset=target_cols)  # CHANGED: use target_cols defined above
                if len(df) == 0:
                    continue
                drop_cols = ["sales", "d_num", "time_idx"] + target_cols
                if features is None:
                    features = [
                        c for c in df.columns
                        if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])
                    ]
                    print(f"Features fixed ({len(features)} features)")
                scaler.partial_fit(df[features].values.astype(np.float32))

            # CHANGED: use self.cat_maps (set in preprocess) instead of local cat_maps
            n_items = len(self.cat_maps["item_id"])
            print("Scaler fitted")

        joblib.dump(features,        f"{self.output_dir}/features.pkl")
        joblib.dump(scaler,          f"{self.output_dir}/scaler.pkl")
        joblib.dump(n_items,         f"{self.output_dir}/n_items.pkl")
        joblib.dump(self.cat_maps,   f"{self.output_dir}/cat_maps.pkl")  # CHANGED: self.cat_maps

        # NEW: pre-train NN for 3 epochs then freeze — replaces 1 gradient step per chunk
        model_nn = FeatureNN(n_items, len(features)).to(self.device)
        if os.path.exists(checkpoint_path):
            model_nn.load_state_dict(torch.load(f"{self.output_dir}/checkpoint_nn.pth"))
            for p in model_nn.parameters():           # NEW: freeze on resume too
                p.requires_grad_(False)
            model_nn.eval()
            print("Loaded and froze checkpoint NN.")
        else:
            # REMOVED: opt = torch.optim.Adam(...) — optimiser now lives in _pretrain_nn
            model_nn = self._pretrain_nn(model_nn, scaler, features, target_cols, n_epochs=3)

        torch.save(model_nn.state_dict(), f"{self.output_dir}/nn_embedding.pth")

        # NEW: compute val embeddings once outside all loops (safe because NN is frozen)
        print("Computing val embeddings...")
        val_df     = self.val_processed.dropna(subset=target_cols)
        X_val      = scaler.transform(val_df[features].values.astype(np.float32))
        X_val_t    = torch.tensor(X_val,                     dtype=torch.float32).to(self.device)
        item_val_t = torch.tensor(val_df["item_id"].values,  dtype=torch.long   ).to(self.device)
        with torch.no_grad():
            _, emb_val = model_nn(X_val_t, item_val_t)
        X_val_aug = np.concatenate([X_val, emb_val.cpu().numpy()], axis=1)
        print(f"Val embeddings ready. Val rows: {len(val_df)}")

        # NEW: GPU params extracted to a shared dict — replaces inline dict repeated 252x
        lgb_params_base = {
            "objective"      : "quantile",
            "num_leaves"     : 64,
            "learning_rate"  : 0.05,
            "verbose"        : -1,
            "device"         : "gpu",       # NEW: was missing — LightGBM ran on CPU before
            "gpu_platform_id": 0,
            "gpu_device_id"  : 0,
        }

        start_time = time.time()

        for chunk_idx, i in enumerate(range(0, total_rows, chunk_size)):
            if chunk_idx < start_chunk:
                continue

            # REMOVED: .copy() — dropna already returns a new frame
            df = df_all.iloc[i:i+chunk_size].dropna(subset=target_cols)  # CHANGED

            # REMOVED: create_targets(df), encode_categories(df), inf/NaN cleaning loop
            # Already done in preprocess()

            progress = (chunk_idx + 1) / total_chunks * 100
            elapsed  = time.time() - start_time
            avg_time = elapsed / max(chunk_idx - start_chunk + 1, 1)
            remaining= avg_time * (total_chunks - chunk_idx - 1)
            print(f"\nChunk {chunk_idx+1}/{total_chunks} ({progress:.1f}%) | ETA {remaining/60:.1f} min")

            if len(df) == 0:
                continue

            # CHANGED: torch.no_grad() — no backprop since NN is frozen
            X      = scaler.transform(df[features].values.astype(np.float32))
            X_t    = torch.tensor(X,                    dtype=torch.float32).to(self.device)
            item_t = torch.tensor(df["item_id"].values, dtype=torch.long   ).to(self.device)
            # REMOVED: y_nn, pred, loss, opt.zero_grad(), loss.backward(), opt.step()
            with torch.no_grad():
                _, emb = model_nn(X_t, item_t)
            X_aug = np.concatenate([X, emb.cpu().numpy()], axis=1)

            # CHANGED: loop order swapped — h outer, q inner so dataset is built once per horizon
            # REMOVED: outer quantile loop was first — caused 9x redundant dataset construction
            for h in range(1, 29):
                print(f"   → h {h}/28", end="\r")

                X_h    = np.concatenate(
                    [X_aug, np.full((len(X_aug), 1), h / 28, dtype=np.float32)], axis=1
                ).astype(np.float32)
                y_h    = df[f"target_{h}"].values.astype(np.float32)
                dtrain = lgb.Dataset(X_h, y_h, free_raw_data=False)  # CHANGED: built once per h, not per (q, h)

                for q in quantiles:  # CHANGED: quantile loop now inner
                    models[q] = lgb.train(
                        {**lgb_params_base, "alpha": q},          # CHANGED: uses shared GPU params dict
                        dtrain,
                        num_boost_round=rounds_per_chunk,          # CHANGED: was hardcoded 200 — now budget-controlled
                        init_model=models[q]
                    )

            print(f"\nCheckpointing chunk {chunk_idx+1}...")
            save_path = self.output_dir
            for q in quantiles:
                models[q].save_model(f"{save_path}/checkpoint_lgb_q_{q}.txt")
                models[q].save_model(f"{save_path}/lgb_q_{q}.txt")
            torch.save(model_nn.state_dict(),  f"{save_path}/checkpoint_nn.pth")
            joblib.dump(scaler,                f"{save_path}/checkpoint_scaler.pkl")
            joblib.dump(features,              f"{save_path}/checkpoint_features.pkl")
            joblib.dump(n_items,               f"{save_path}/checkpoint_n_items.pkl")
            joblib.dump(chunk_idx,             f"{save_path}/checkpoint_chunk.pkl")
            joblib.dump(self.cat_maps,         f"{save_path}/checkpoint_cat_maps.pkl")  # CHANGED: self.cat_maps
            torch.save(model_nn.state_dict(),  f"{save_path}/nn_embedding.pth")
            joblib.dump(scaler,                f"{save_path}/scaler.pkl")
            joblib.dump(features,              f"{save_path}/features.pkl")
            joblib.dump(n_items,               f"{save_path}/n_items.pkl")
            joblib.dump(self.cat_maps,         f"{save_path}/cat_maps.pkl")  # CHANGED: self.cat_maps

        self.models   = models
        self.features = features
        self.scaler   = scaler

    def predict(self):
        print("Loading models...")
        quantiles = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]
        models = {}
        for q in quantiles:
            path = f"{self.output_dir}/lgb_q_{q}.txt"
            models[q] = lgb.Booster(model_file=path)

        scaler       = joblib.load(f"{self.output_dir}/scaler.pkl")
        self.features= joblib.load(f"{self.output_dir}/features.pkl")
        n_items      = joblib.load(f"{self.output_dir}/n_items.pkl")

        model_nn = FeatureNN(n_items, len(self.features))
        model_nn.load_state_dict(torch.load(f"{self.output_dir}/nn_embedding.pth"))
        model_nn.eval()
        model_nn = model_nn.to(self.device)

        # load test data
        df = self.test_processed.copy()
        df = df[(df["d_num"] >= 1886) & (df["d_num"] <= 1913)]

        # REMOVED: cat re-encoding block (lines 644-649)
        # test_processed is already encoded in preprocess() — re-mapping would corrupt item_id to -1

        X      = scaler.transform(df[self.features].values.astype(np.float32))
        X_t    = torch.tensor(X,                    dtype=torch.float32).to(self.device)
        item_t = torch.tensor(df["item_id"].values, dtype=torch.long   ).to(self.device)  # CHANGED: item -> item_t

        with torch.no_grad():
            _, emb = model_nn(X_t, item_t)  # CHANGED: item -> item_t
            emb    = emb.cpu().numpy()

        X_aug    = np.concatenate((X, emb), axis=1)
        last_df  = df.groupby("id").tail(1).reset_index(drop=True)
        X_base   = X_aug[-len(last_df):]
        n_series = len(last_df)
        preds    = np.zeros((n_series, 28, len(quantiles)), dtype=np.float32)

        print("Generating forecasts")
        for qi, q in enumerate(quantiles):
            model = models[q]
            for h in range(1, 29):
                X_h = np.concatenate(
                    (X_base, np.full((n_series, 1), h / 28, dtype=np.float32)),  # CHANGED: h -> h/28 to match training
                    axis=1
                )
                preds[:, h-1, qi] = model.predict(X_h)

        # convert to dataframe
        rows = []
        for i, row in last_df.iterrows():
            for h in range(28):
                rec = {"id": row["id"], "day_ahead": h + 1}
                for qi, q in enumerate(quantiles):
                    rec[f"q{q}"] = preds[i, h, qi]
                rows.append(rec)

        preds_df = pd.DataFrame(rows)
        preds_df = preds_df.sort_values(["id", "day_ahead"]).reset_index(drop=True)
        print("Fixing quantiles")
        q_cols = [f"q{q}" for q in quantiles]
        preds_df[q_cols] = preds_df[q_cols].clip(lower=0)
        preds_df[q_cols] = np.maximum.accumulate(preds_df[q_cols].values, axis=1)

        save_path = f"{self.output_dir}/{self.model_name}_predictions.csv"
        preds_df.to_csv(save_path, index=False)
        print(f"Saved predictions → {save_path}")

        return preds_df

In [14]:
model = LightGBM_NN()
model.load_and_split_data()
model.preprocess()

# manually replicate the scaler-fitting step so we can benchmark
# only the chunk training loop itself
target_cols   = [f"target_{h}" for h in range(1, 29)]
df_all        = model.train_processed
chunk_size    = 1_000_000
total_rows    = len(df_all)
total_chunks  = int(np.ceil(total_rows / chunk_size))
quantiles     = model.QUANTILES

# fit scaler on first 3 chunks only (just for benchmark)
print("Fitting scaler on first 3 chunks...")
scaler   = StandardScaler()
features = None
for i in range(0, min(3 * chunk_size, total_rows), chunk_size):
    df = df_all.iloc[i:i+chunk_size].dropna(subset=target_cols)
    if features is None:
        drop_cols = ["sales", "d_num", "time_idx"] + target_cols
        features  = [c for c in df.columns
                     if c not in drop_cols
                     and df[c].dtype in [np.float32, np.float64, np.int16, np.int8]]
    scaler.partial_fit(df[features].values.astype(np.float32))

n_items  = len(model.cat_maps["item_id"])
# pre-train NN (same as full run — not skipped)
print("Pre-training NN...")
nn_pretrain_start = time.time()
model_nn = FeatureNN(n_items, len(features)).to(model.device)
model_nn = model._pretrain_nn(model_nn, scaler, features, target_cols, n_epochs=3)
nn_pretrain_time = time.time() - nn_pretrain_start
print(f"NN pre-train time: {nn_pretrain_time/60:.1f} min")

# val embeddings (same as full run)
val_df     = model.val_processed.dropna(subset=target_cols)
X_val      = scaler.transform(val_df[features].values.astype(np.float32))
X_val_t    = torch.tensor(X_val,                    dtype=torch.float32).to(model.device)
item_val_t = torch.tensor(val_df["item_id"].values, dtype=torch.long   ).to(model.device)
with torch.no_grad():
    _, emb_val = model_nn(X_val_t, item_val_t)
X_val_aug = np.concatenate([X_val, emb_val.cpu().numpy()], axis=1)

total_tree_budget = 500
rounds_per_chunk  = max(1, total_tree_budget // total_chunks)

lgb_params_base = {
    "objective"      : "quantile",
    "num_leaves"     : 64,
    "learning_rate"  : 0.05,
    "verbose"        : -1,
    "device"         : "gpu",
    "gpu_platform_id": 0,
    "gpu_device_id"  : 0,
}

# --- benchmark 3 chunks ---
N_BENCHMARK = 3
models      = {q: None for q in quantiles}
chunk_times = []

print(f"\nBenchmarking {N_BENCHMARK} chunks "
      f"({rounds_per_chunk} rounds/chunk, {len(quantiles)} quantiles x 28 horizons)...\n")

for chunk_idx in range(N_BENCHMARK):
    i  = chunk_idx * chunk_size
    df = df_all.iloc[i:i+chunk_size].dropna(subset=target_cols)
    if len(df) == 0:
        continue

    t0 = time.time()

    # NN embeddings
    X      = scaler.transform(df[features].values.astype(np.float32))
    X_t    = torch.tensor(X,                    dtype=torch.float32).to(model.device)
    item_t = torch.tensor(df["item_id"].values, dtype=torch.long   ).to(model.device)
    with torch.no_grad():
        _, emb = model_nn(X_t, item_t)
    X_aug = np.concatenate([X, emb.cpu().numpy()], axis=1)

    # LightGBM training
    for h in range(1, 29):
        X_h    = np.concatenate(
            [X_aug, np.full((len(X_aug), 1), h / 28, dtype=np.float32)], axis=1
        ).astype(np.float32)
        y_h    = df[f"target_{h}"].values.astype(np.float32)
        dtrain = lgb.Dataset(X_h, y_h, free_raw_data=False)
        for q in quantiles:
            models[q] = lgb.train(
                {**lgb_params_base, "alpha": q},
                dtrain,
                num_boost_round=rounds_per_chunk,
                init_model=models[q]
            )

    elapsed = time.time() - t0
    chunk_times.append(elapsed)
    print(f"Chunk {chunk_idx+1}: {elapsed/60:.2f} min")

# --- extrapolate ---
avg_chunk   = np.mean(chunk_times)
# later chunks are slower due to growing model — use last chunk as conservative estimate
worst_chunk = chunk_times[-1]

flat_estimate      = avg_chunk    * total_chunks
worst_estimate     = worst_chunk  * total_chunks

print(f"\n{'='*45}")
print(f"Chunks benchmarked  : {N_BENCHMARK}")
print(f"Rounds per chunk    : {rounds_per_chunk}")
print(f"Total chunks        : {total_chunks}")
print(f"Avg chunk time      : {avg_chunk/60:.2f} min")
print(f"Slowest chunk time  : {worst_chunk/60:.2f} min")
print(f"\nFlat estimate       : {flat_estimate/3600:.2f} hrs  (optimistic)")
print(f"Conservative est.   : {worst_estimate/3600:.2f} hrs  (pessimistic)")
print(f"NN pre-train (fixed): {nn_pretrain_time/60:.1f} min")
print(f"\nVerify LightGBM GPU active — should say 'gpu' below:")
print(models[quantiles[0]].params.get("device", "NOT SET — running on CPU!"))

Loaded cached data splits.
starting preprocessing
Fitting scaler on first 3 chunks...
Pre-training NN...
  NN pretrain epoch 1/3 | avg loss 9.8722
  NN pretrain epoch 2/3 | avg loss 8.4958
  NN pretrain epoch 3/3 | avg loss 6.1937
NN frozen.
NN pre-train time: 1.4 min

Benchmarking 3 chunks (9 rounds/chunk, 9 quantiles x 28 horizons)...

Chunk 1: 4.11 min
Chunk 2: 7.77 min
Chunk 3: 12.53 min

Chunks benchmarked  : 3
Rounds per chunk    : 9
Total chunks        : 55
Avg chunk time      : 8.14 min
Slowest chunk time  : 12.53 min

Flat estimate       : 7.46 hrs  (optimistic)
Conservative est.   : 11.49 hrs  (pessimistic)
NN pre-train (fixed): 1.4 min

Verify LightGBM GPU active — should say 'gpu' below:
gpu
